# CS 521 HW4, Problem 2: Certified Training for Multiple Norms (CURE)

Open this notebook in Google Colab and select a GPU runtime (*Runtime > Change runtime type > T4 GPU*).

The data, the model, the attacks, the verifier, the evaluation and the default hyperparameters are not changed except where the homework asks for it.

**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]

device: cuda


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [ ]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 484kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.8MB/s]


torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [ ]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [ ]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [ ]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [ ]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [ ]:
LAM = {"linf": 0.7, "l2": 0.7}


def propagation_region(model, x, y, eps, lam, norm):
    # Returns (center, tau): x* from PGD in the (eps - tau)-ball around x, and the radius tau of the small ball.
    tau = lam * eps
    model.eval()
    x_star = pgd(model, x, y, eps - tau, norm, steps=5)
    model.train()
    return x_star, tau


def small_ball_z(model, x, y, eps, lam, norm):
    # z_hat = -m, where m are the IBP lower bounds of the margins f_y - f_j on the small ball around x*.
    x_star, tau = propagation_region(model, x, y, eps, lam, norm)
    return -ibp_margin_bounds(model, x_star, tau, norm, y)


def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    return F.cross_entropy(small_ball_z(model, x, y, eps, lam, norm), y, reduction=reduction)


def single_norm_loss(model, x, y, t, norm):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    return certified_loss(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)

In [ ]:
models, results = {}, []
for name, loss_fn in [("standard", standard_loss),
                      ("linf", lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf")),
                      ("l2", lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"))]:
    models[name] = make_model()
    sec = train(models[name], loss_fn)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5738  time 1 s
epoch  2  loss 0.2029  time 1 s
epoch  3  loss 0.1459  time 1 s
epoch  4  loss 0.1221  time 1 s
epoch  5  loss 0.1030  time 2 s
epoch  6  loss 0.0907  time 2 s
epoch  7  loss 0.0805  time 2 s
epoch  8  loss 0.0672  time 2 s
epoch  9  loss 0.0656  time 3 s
epoch 10  loss 0.0553  time 3 s
epoch 11  loss 0.0510  time 3 s
epoch 12  loss 0.0484  time 3 s
epoch 13  loss 0.0421  time 4 s
epoch 14  loss 0.0388  time 4 s
epoch 15  loss 0.0351  time 4 s
epoch 16  loss 0.0182  time 4 s
epoch 17  loss 0.0138  time 4 s
epoch 18  loss 0.0126  time 5 s
epoch 19  loss 0.0117  time 5 s
epoch 20  loss 0.0112  time 5 s
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4870  time 5 s
epoch  3  loss 0.4765  time 7 s
epoch  4  loss 0.4804  time 9 s
epoch  5  loss 0.4975  time 10 s
epoch  6  loss 0.5255  time 11 s
epoch  7  loss 0.5673  time 13 s
epoch  8  loss 0.6250  time 14 s
epoch  9  loss 0.6905  time 15 s
epoch 10  loss 0.7668  time 16 s
epoch 11  loss 0.8412  time 18 s
e

# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [ ]:
def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    s = eps_scale(t)
    # z_hat of both norms on their small balls (the same values as in certified_loss)
    z_inf = small_ball_z(model, x, y, s * EPS["linf"], LAM["linf"], "linf")
    z_2 = small_ball_z(model, x, y, s * EPS["l2"], LAM["l2"], "l2")
    ce_inf = F.cross_entropy(z_inf, y, reduction="none")
    ce_2 = F.cross_entropy(z_2, y, reduction="none")
    loss = torch.max(ce_inf, ce_2).mean()                       # CURE-Max
    if eta > 0:
        # gamma: images certified for l2 on the small ball, i.e. z_hat_j < 0 for all j != y (z_hat_y = 0)
        is_y = F.one_hot(y, z_2.shape[1]).bool()
        gamma = ((z_2 < 0) | is_y).all(dim=1)
        if gamma.any():
            # KL(d2 || d_inf) with d^p = softmax(z_hat^p), averaged over gamma
            kl = F.kl_div(F.log_softmax(z_inf[gamma], dim=1), F.softmax(z_2[gamma], dim=1), reduction="batchmean")
            loss = loss + eta * kl                              # bound alignment (BA)
    return loss


def gradient_projection(g_n, g_c, beta):
    # beta * GP(g_n, g_c) + (1 - beta) * g_c for one parameter tensor
    cos = F.cosine_similarity(g_n.flatten(), g_c.flatten(), dim=0)
    gp = cos.clamp(min=0) * g_n          # cos * g_n if cos > 0, else 0
    return beta * gp + (1 - beta) * g_c

In [ ]:
for name, loss_fn, beta in [("cure_max", lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
                            ("cure_max_ba", cure_loss, None),
                            ("cure_max_ba_gp", cure_loss, 0.5)]:
    runs = []
    for seed in [0, 1]:
        torch.manual_seed(seed)   # the seed also sets the initial weights
        models[f"{name}_seed{seed}"] = make_model()
        sec = train(models[f"{name}_seed{seed}"], loss_fn, seed=seed, gp_beta=beta)
        runs.append(dict(evaluate(models[f"{name}_seed{seed}"], name, attacks=True), time=sec))
    mean = {"name": name}
    for k in runs[0]:
        if k != "name":
            mean[k] = sum(r[k] for r in runs) / len(runs)
    results.append(mean)
print_results(results)

epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4887  time 3 s
epoch  3  loss 0.5010  time 5 s
epoch  4  loss 0.5257  time 8 s
epoch  5  loss 0.5658  time 11 s
epoch  6  loss 0.6304  time 14 s
epoch  7  loss 0.7264  time 16 s
epoch  8  loss 0.8331  time 19 s
epoch  9  loss 0.9535  time 21 s
epoch 10  loss 1.0763  time 24 s
epoch 11  loss 1.2127  time 27 s
epoch 12  loss 1.2672  time 30 s
epoch 13  loss 1.2403  time 32 s
epoch 14  loss 1.2206  time 35 s
epoch 15  loss 1.2015  time 38 s
epoch 16  loss 1.1533  time 40 s
epoch 17  loss 1.1431  time 43 s
epoch 18  loss 1.1379  time 46 s
epoch 19  loss 1.1368  time 48 s
epoch 20  loss 1.1341  time 51 s
epoch  1  loss 0.5640  time 0 s
epoch  2  loss 0.4955  time 3 s
epoch  3  loss 0.5016  time 5 s
epoch  4  loss 0.5431  time 8 s
epoch  5  loss 0.6039  time 11 s
epoch  6  loss 0.6793  time 14 s
epoch  7  loss 0.7705  time 16 s
epoch  8  loss 0.8708  time 19 s
epoch  9  loss 0.9819  time 22 s
epoch 10  loss 1.1123  time 24 s
epoch 11  loss 1.2

# (c) Your method

Write the loss of your method in `my_loss`. Train it with `train` and the default settings, as for the other methods.
If your method needs a different update, write your own copy of `train`.
Do not change the verifier or `evaluate`.

## (c)(i) Idea: CROWN-IBP bounds inside CURE

The loss of CURE-Max + BA uses IBP bounds on the SABR small ball, and IBP bounds are loose. CROWN-IBP [1] computes the bounds of the hidden layers with IBP and the bounds of the final margins with one backward CROWN pass, which can give tighter margin bounds than IBP at a lower cost than full CROWN. In [1], it is used for $\ell_\infty$ on the ball around the input.

My method uses it on the SABR small ball of both norms, and takes the element-wise maximum with the IBP bound. The rest is as in CURE-Max + BA ($\eta = 0.5$): the PGD center $x^*_p$ and the radius $\tau_p = \lambda\epsilon_p$, the per-image maximum of the two losses (CURE-Max) and the BA term.

```
for p in {linf, l2}:
    x*, tau = propagation_region(model, x, y, scale(t) * EPS[p], LAM[p], p)    # SABR centre, same as (a)
    m_crown = CROWN-IBP lower margin bound on B_p(x*, tau)                      # IBP hidden bounds + backward pass
    m_ibp   = IBP lower margin bound on B_p(x*, tau)
    m       = max(m_crown, m_ibp)                  # both are sound, so the element-wise max is sound
    z_hat[p] = -m
```

$\gamma$ and $d^p = \mathrm{softmax}(\hat z^p)$ are computed from the new $\hat z^p$. For $\ell_2$, the backward pass uses $\min_{\|x'-c\|_2\le r} a^\top x' = a^\top c - r\|a\|_2$ at the input.
The ablation `mine_crownonly` removes the max with IBP (`use_ibp=False`).

I expect a higher union certified accuracy for two reasons. Both bounds are lower bounds of the true margin, so a tighter bound gives a loss closer to the loss at the worst case in the ball. The verifier also certifies with the larger of the IBP and CROWN bounds, so the network is trained on the bound that is used at test time.

**References**

* [1] H. Zhang, H. Chen, C. Xiao, S. Gowal, R. Stanforth, B. Li, D. Boning, C.-J. Hsieh. *Towards Stable and Efficient Training of Verifiably Robust Neural Networks* (CROWN-IBP). ICLR 2020. arXiv:1906.06316.


In [ ]:
def crown_ibp_margin_bounds(model, c, r, norm, y):
    # CROWN-IBP: IBP bounds for all hidden pre-activations, then one CROWN backward pass for the margins f_y - f_j.
    layers = linear_layers(model)
    bounds = [first_layer_bounds(layers[0], c, r, norm)]
    l, u = bounds[0]
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
        bounds.append((l, u))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


def my_loss(model, x, y, t, eta=0.5, use_ibp=True):
    # CURE-Max + BA, but z_hat comes from the larger of the IBP and CROWN-IBP lower bounds (like the verifier).
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    s = eps_scale(t)
    z = {}
    for norm in ["linf", "l2"]:
        x_star, tau = propagation_region(model, x, y, s * EPS[norm], LAM[norm], norm)
        m = crown_ibp_margin_bounds(model, x_star, tau, norm, y)
        if use_ibp:
            m = torch.maximum(m, ibp_margin_bounds(model, x_star, tau, norm, y))
        z[norm] = -m
    ce_inf = F.cross_entropy(z["linf"], y, reduction="none")
    ce_2 = F.cross_entropy(z["l2"], y, reduction="none")
    loss = torch.max(ce_inf, ce_2).mean()
    is_y = F.one_hot(y, 10).bool()
    gamma = ((z["l2"] < 0) | is_y).all(dim=1)
    if eta > 0 and gamma.any():
        loss = loss + eta * F.kl_div(F.log_softmax(z["linf"][gamma], dim=1), F.softmax(z["l2"][gamma], dim=1),
                                     reduction="batchmean")
    return loss

In [ ]:
for name, loss_fn in [("mine", lambda m, x, y, t: my_loss(m, x, y, t)),
                      ("mine_crownonly", lambda m, x, y, t: my_loss(m, x, y, t, use_ibp=False))]:
    runs = []
    for seed in [0, 1]:
        tag = f"{name}_seed{seed}"
        torch.manual_seed(seed)   # the seed also sets the initial weights
        models[tag] = make_model()
        sec = train(models[tag], loss_fn, seed=seed)
        runs.append(dict(evaluate(models[tag], tag, attacks=True), time=sec))
    print_results(runs)   # one row per seed
    mean = {"name": name}
    for k in runs[0]:
        if k != "name":
            mean[k] = sum(r[k] for r in runs) / len(runs)
    results.append(mean)
print_results(results)

epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.2745  time 5 s
epoch  3  loss 0.2975  time 10 s
epoch  4  loss 0.3349  time 14 s
epoch  5  loss 0.3889  time 19 s
epoch  6  loss 0.4400  time 23 s
epoch  7  loss 0.5198  time 28 s
epoch  8  loss 0.5982  time 33 s
epoch  9  loss 0.7162  time 37 s
epoch 10  loss 0.8197  time 42 s
epoch 11  loss 0.9286  time 47 s
epoch 12  loss 0.9941  time 51 s
epoch 13  loss 0.9703  time 56 s
epoch 14  loss 0.9565  time 60 s
epoch 15  loss 0.9535  time 65 s
epoch 16  loss 0.8943  time 70 s
epoch 17  loss 0.8857  time 74 s
epoch 18  loss 0.8821  time 78 s
epoch 19  loss 0.8806  time 83 s
epoch 20  loss 0.8779  time 88 s
epoch  1  loss 0.5640  time 0 s
epoch  2  loss 0.2682  time 5 s
epoch  3  loss 0.2925  time 9 s
epoch  4  loss 0.3285  time 14 s
epoch  5  loss 0.3956  time 19 s
epoch  6  loss 0.4548  time 23 s
epoch  7  loss 0.5466  time 28 s
epoch  8  loss 0.6411  time 33 s
epoch  9  loss 0.7326  time 37 s
epoch 10  loss 0.8515  time 42 s
epoch 11  loss 

In [ ]:
cure_names = ["cure_max", "cure_max_ba", "cure_max_ba_gp"]
row = lambda n: next(r for r in results if r["name"] == n)
best = max(cure_names, key=lambda n: row(n)["union"])
eta = 0.0 if best == "cure_max" else 0.5
beta = 0.5 if best == "cure_max_ba_gp" else None
epochs = round(20 * row("mine")["time"] / row(best)["time"])
print(f"best CURE method of (b): {best};  CURE trains for {epochs} epochs to use the same time as mine (20 epochs)")

runs = []
for seed in [0, 1]:
    tag = f"{best}_equal_time_seed{seed}"
    torch.manual_seed(seed)
    models[tag] = make_model()
    sec = train(models[tag], lambda m, x, y, t: cure_loss(m, x, y, t, eta=eta), epochs=epochs, seed=seed, gp_beta=beta)
    runs.append(dict(evaluate(models[tag], tag, attacks=True), time=sec))
print_results(runs)   # one row per seed
mean = {"name": f"{best}_equal_time"}
for k in runs[0]:
    if k != "name":
        mean[k] = sum(r[k] for r in runs) / len(runs)
results.append(mean)
print("\nSummary (means over the two seeds):")
print_results(results)

best CURE method of (b): cure_max;  CURE trains for 34 epochs to use the same time as mine (20 epochs)
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4887  time 3 s
epoch  3  loss 0.5010  time 5 s
epoch  4  loss 0.5257  time 8 s
epoch  5  loss 0.5658  time 11 s
epoch  6  loss 0.6304  time 14 s
epoch  7  loss 0.7264  time 16 s
epoch  8  loss 0.8331  time 19 s
epoch  9  loss 0.9535  time 22 s
epoch 10  loss 1.0763  time 24 s
epoch 11  loss 1.2127  time 27 s
epoch 12  loss 1.2672  time 30 s
epoch 13  loss 1.2403  time 32 s
epoch 14  loss 1.2206  time 35 s
epoch 15  loss 1.2015  time 38 s
epoch 16  loss 1.1533  time 40 s
epoch 17  loss 1.1431  time 43 s
epoch 18  loss 1.1379  time 46 s
epoch 19  loss 1.1368  time 49 s
epoch 20  loss 1.1341  time 51 s
epoch 21  loss 1.1336  time 54 s
epoch 22  loss 1.1290  time 56 s
epoch 23  loss 1.1287  time 59 s
epoch 24  loss 1.1288  time 62 s
epoch 25  loss 1.1244  time 65 s
epoch 26  loss 1.1261  time 67 s
epoch 27  loss 1.1221  time 70 s
epoch 28  

# Report: answers

All numbers (in %) are from `evaluate` on the 10,000 test images, with $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$. Union = certified for both norms. Time is in seconds per epoch.

## (a) Single-norm certified training

One run (seed 0) for each network.

| Network | Clean | linf cert | l2 cert | Union cert | PGD linf | PGD l2 | PGD union | Time |
|---|---|---|---|---|---|---|---|---|
| Standard | 97.52 | 0.00 | 0.00 | 0.00 | 0.64 | 8.47 | 0.63 | 0.3 |
| linf certified | 93.52 | 80.88 | 5.51 | 5.51 | 84.63 | 71.43 | 71.20 | 1.5 |
| l2 certified | 93.14 | 54.57 | 58.23 | 51.70 | 68.58 | 78.14 | 68.14 | 1.4 |

**(i) Transfer.** The transfer is only in one direction.

* The $\ell_\infty$ network is certified for $\ell_\infty$ at 80.88 but for $\ell_2$ at only 5.51. Its PGD accuracy for $\ell_2$ is 71.43, so the $\ell_2$ robustness is largely there empirically, but the verifier cannot prove it (gap 65.92).
* The $\ell_2$ network is certified for $\ell_\infty$ at 54.57 (PGD 68.58) and for $\ell_2$ at 58.23 (PGD 78.14).
* The standard network is not certified for any norm (0.00). Its PGD accuracy is 0.64 for $\ell_\infty$, 8.47 for $\ell_2$ and 0.63 for the union.

**(ii) Norm that limits the union.**

* $\ell_\infty$ network: $\ell_2$ (union 5.51 = $\ell_2$ certified).
* $\ell_2$ network: $\ell_\infty$ (54.57 vs 58.23). The union (51.70) is lower than both. Since union = $\ell_\infty$ + $\ell_2$ − (certified for at least one norm), 6.53% of the images are certified for $\ell_2$ only (58.23 − 51.70) and 2.87% for $\ell_\infty$ only (54.57 − 51.70).
* Standard network: both norms (0.00).

## (b) CURE

Mean of two seeds (0 and 1) for the CURE rows. The first three rows are from part (a) (one run, seed 0).

| Network | Clean | linf cert | l2 cert | Union cert | PGD linf | PGD l2 | PGD union | Time |
|---|---|---|---|---|---|---|---|---|
| Standard | 97.52 | 0.00 | 0.00 | 0.00 | 0.64 | 8.47 | 0.63 | 0.3 |
| linf certified | 93.52 | 80.88 | 5.51 | 5.51 | 84.63 | 71.43 | 71.20 | 1.5 |
| l2 certified | 93.14 | 54.57 | 58.23 | 51.70 | 68.58 | 78.14 | 68.14 | 1.4 |
| CURE-Max | 92.44 | 67.61 | 57.18 | **57.09** | 77.23 | 78.44 | 76.59 | 2.6 |
| CURE-Max + BA | 92.06 | 67.25 | 57.09 | 57.06 | 76.09 | 77.77 | 75.59 | 2.7 |
| CURE-Max + BA + GP | 92.49 | 67.05 | 56.55 | 56.48 | 76.31 | 77.84 | 75.77 | 2.8 |

**(i) CURE vs single-norm.**

* Highest union certified accuracy: CURE-Max (57.09), above the $\ell_2$ network (51.70) and the $\ell_\infty$ network (5.51).
* Compared with the single-norm networks, CURE-Max has a lower $\ell_\infty$ certified accuracy than the $\ell_\infty$ network (67.61 vs 80.88) and a similar $\ell_2$ certified accuracy to the $\ell_2$ network (57.18 vs 58.23).
* Highest clean accuracy: the standard network (97.52), which has no certified robustness. Among the certified networks, the $\ell_\infty$ network is the highest (93.52). Among the CURE variants, CURE-Max + BA + GP is the highest (92.49), and the three variants are within 0.5 of each other.

**(ii) BA and GP.** BA and GP do not increase the union certified accuracy in our runs. The union is 57.09 (CURE-Max), 57.06 (+BA) and 56.48 (+BA+GP). Possible reasons:

* BA acts only on the images in $\gamma$, which are already certified for $\ell_2$ on the small ball, and the max already puts the loss on the harder norm of each image.
* GP is applied only for $t \ge 11$ (the last 9 epochs). It mixes in the gradient of the clean loss, which gives a higher clean accuracy than BA (92.49 vs 92.06) and not a higher certified accuracy.

**(iii) Time.** CURE uses 2.6–2.8 s per epoch, the single-norm networks 1.4–1.5 s, i.e. about 1.7–2.0x (standard training: 0.3 s). CURE runs the PGD search and the IBP pass for both norms, and a single-norm step runs them for one norm. BA adds 0.1 s, and GP adds 0.1 s more for the extra backward pass on the clean loss.


## (c)(ii) Implementation

`crown_ibp_margin_bounds` and `my_loss` above. Training uses `train` with the default settings (20 epochs, same schedule), seeds 0 and 1. `spec_matrix` and `_backward_lower` are the CROWN functions of the verifier (not changed).

## (c)(iii) Results

Mean of two seeds. "Equal time" is CURE-Max trained for 34 epochs, which takes about the same total time as my method for 20 epochs (about 88 s each). CURE-Max is the best network of part (b).

| Method | Clean | linf cert | l2 cert | Union cert | PGD union | Time | Epochs |
|---|---|---|---|---|---|---|---|
| CURE-Max (best of (b)) | 92.44 | 67.61 | 57.18 | 57.09 | 76.59 | 2.6 | 20 |
| CURE-Max, equal time | 92.56 | 67.56 | 57.01 | 56.92 | 76.64 | 2.6 | 34 |
| **Mine** | **94.59** | **72.19** | **64.09** | **64.06** | 77.98 | 4.4 | 20 |

Per seed, my method has union 64.24 (seed 0) and 63.89 (seed 1) with clean 94.86 and 94.31; CURE-Max for 34 epochs has union 57.48 and 56.36. Compared with CURE-Max, my method has +6.97% union and +2.15% clean, with 1.7x the time per epoch. More epochs do not help CURE (56.92 at equal time).

## (c)(iv) Ablation

Mine uses BA with $\eta = 0.5$, so the ablation keeps BA and removes only the CROWN-IBP bound (the third row is CURE-Max + BA of part (b)).

| Variant | Clean | linf cert | l2 cert | Union cert | Time |
|---|---|---|---|---|---|
| Mine (CROWN-IBP, max with IBP) | 94.59 | 72.19 | 64.09 | 64.06 | 4.4 |
| Without the max with IBP (CROWN-IBP only, `mine_crownonly`) | 94.60 | 72.28 | 64.09 | 64.06 | 3.5 |
| Without CROWN-IBP (IBP only, = CURE-Max + BA) | 92.06 | 67.25 | 57.09 | 57.06 | 2.7 |

Removing the max with IBP changes nothing (union 64.06 for both) and saves 0.9 s per epoch, so the max with IBP is not helpful. Removing CROWN-IBP costs 7.00% union. BA does not explain the gain, since CURE-Max without BA has 57.09 (part (b)).

## (c)(v) Analysis

The method works.

* The union goes from 57.09 to 64.06, and both norms improve: $\ell_\infty$ 67.61 → 72.19 (+4.58) and $\ell_2$ 57.18 → 64.09 (+6.91). The clean accuracy improves from 92.44 to 94.59. The union difference is larger than the difference between seeds (0.35 for my method, 1.12 for CURE-Max), and the equal-time run shows that it is not due to compute.
* $\ell_2$ limits the union in both cases: 64.06 vs 64.09 for my method, and 57.09 vs 57.18 for CURE-Max.
* The difference between the PGD union and the certified union decreases from 19.50 (76.59 − 57.09) to 13.92 (77.98 − 64.06), while the PGD union increases by only 1.39. A smaller difference means tighter bounds.
* Why it works: the ablation shows that the gain comes from the CROWN-IBP bound in the loss. Removing it (IBP only, same BA) costs 7.00% union, and removing the max with IBP changes nothing.
